In [1]:
# from fastapi import FastAPI
# from pydantic import BaseModel
import pandas as pd
# import joblib
# from sklearn.neighbors import NearestNeighbors

In [2]:
df = pd.read_csv("vehicles_dataset.csv")
#df.dropna(inplace=True)
#df = df[df["fuel"] == "Electric"]
#df

In [3]:
#df2 = pd.read_csv('Safercar_data.csv')

# Convert to numeric (in case of 'Not Rated' or strings)
#df2['OVERALL_STARS'] = pd.to_numeric(df2['OVERALL_STARS'], errors='coerce')

# Drop rows with missing or invalid ratings
#df2 = df2.dropna(subset=['OVERALL_STARS'])

# Group by make and calculate the average
#average_ratings = df2.groupby('MAKE')['OVERALL_STARS'].mean().sort_values(ascending=False).to_dict()


#print(average_ratings)


In [4]:
import pandas as pd
import requests
from IPython.display import HTML

# Take input and filter
#make = input("Enter make: ").strip().lower()
#model = input("Enter model: ").strip().lower()
#yr = input("Enter model year: ").strip().lower()

#matches = df2[(df2['MODEL'].str.lower().str.contains(model, na=False)) 
             # & (df2['MODEL_YR'] == int(yr))][['MAKE', 'MODEL', 'MODEL_YR', 'OVERALL_STARS']]
#if matches.shape[0] > 0:
   # display(matches)
#else:
  #  print("Requested car not found")

#display(HTML(f'<a href=https://cars.usnews.com/cars-trucks/{make}/{model}/{yr} target="_blank">{yr} {make.capitalize()} {model.capitalize()}</a>'))

In [15]:
import re

def clean_model_name(model: str) -> str:
    # Define forbidden words
    forbidden_words = {
        "se", "le", "xe", "xle", "ls", "lxs", "ex", "limited", "ultimate", "advanced", "premium", "standard", "range", "touring",
        "sport", "luxury", "pure", "electric", "twin", "performance", "plus", "base", "select", "light", "long", 
        "gt", "a-spec", "edition", "platinum", "sel", "2lt", "lt", "sxt", "sv", "s", "14t", "l", "latitude", "preferred", "active"
    }
    
    # Normalize and split
    words = model.lower().split()
    
    # Filter out forbidden words and numeric tokens (integers or decimals)
    filtered_words = [
        w for w in words
        if w not in forbidden_words and not re.match(r"\d+(\.\d+)", w)
    ]
        
    # Join back into slug
    return "-".join(filtered_words)

# Test cases
#print(clean_model_name("Ioniq 5 SE Standard Range"))                   # → "ioniq-5"
#print(clean_model_name("C40 Recharge Pure Electric Twin Ultimate"))   # → "c40-recharge"
#print(clean_model_name("Electrified GV70 Advanced"))                  # → "electrified-gv70"
#print(clean_model_name("CR-V 1.5T Touring"))                           # → "cr-v"
#print(clean_model_name("CX-30 2.5 S Premium"))     
#print(clean_model_name("GMC Terrain SLE"))

In [37]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# Layout for vertical centering
center_layout = widgets.Layout(
    display="flex",
    flex_flow="column",
    align_items="center",   # centers horizontally
    justify_content="center",
    width="100%"
)

# Dropdowns
min_price_dd = widgets.Dropdown(
    options=[5000, 10000, 15000, 20000, 25000],
    value=10000,
    description="Min Price:"
)

max_price_dd = widgets.Dropdown(
    options=[10000, 20000, 30000, 40000, 50000, 60000],
    value=40000,
    description="Max Price:"
)

fuel_dd = widgets.Dropdown(
    options=["Gasoline", "Diesel", "Hybrid", "Electric"],
    value="Gasoline",
    description="Fuel:"
)

body_dd = widgets.Dropdown(
    options=["Sedan", "SUV", "Truck", "Coupe", "Hatchback", "Convertible"],
    value="SUV",
    description="Body:"
)

rating_dd = widgets.Dropdown(
    options=[1.0, 2.0, 3.0, 4.0, 5.0],
    value=1.0,
    description="Min Rating:"
)

# Button
search_btn = widgets.Button(
    description="Search",
    button_style="primary",
    tooltip="Click to search cars",
    icon="search"
)

# Output area to display results
output = widgets.Output()



# Collect selections
def get_user_input():
    return {
        "min_price": min_price_dd.value,
        "max_price": max_price_dd.value,
        "fuel": fuel_dd.value,
        "body": body_dd.value,
        "min_rating": rating_dd.value,
    }

# Filtering
def filter_cars(df, user_input):
    if user_input["min_price"] > user_input["max_price"]:
        print("Max price should be less than min price")
        return
    df = df[
        (df["price"] >= user_input["min_price"]) &
        (df["price"] <= user_input["max_price"]) &
        (df["fuel"].str.lower() == user_input["fuel"].lower()) &
        (df["body"].str.lower() == user_input["body"].lower()) #&
        #(df["rating"] >= user_input["min_rating"])
    ]
    return df

# Callback for button
def on_search_click(b):    
    with output:
        clear_output()  # clear previous results
        
        user_input = get_user_input()
        target_df = filter_cars(df, user_input)

        visited = set()
        cnt = 0
        
        for item in target_df['name']:
            parts = item.strip().split()
            car = (parts[0], parts[1], " ".join(parts[2:]))
            
            if (parts[0], parts[1]) not in visited:
                make = car[1]
                model = car[2]
                model_url = clean_model_name(model)
                cnt += 1

                display(HTML(f"""
                <center>
                  <a href="https://cars.usnews.com/cars-trucks/{make.lower()}/{model_url}"
                     target="_blank" rel="noopener noreferrer">
                    {make} {model}
                  </a>
                </center>
                """))
            
            visited.add((parts[0], parts[1]))

        if cnt == 0:
            print("Sorry! No vehicles match your requirements")

# Attach callback
search_btn.on_click(on_search_click)

# Wrap all widgets in a vertically centered VBox
ui = widgets.VBox(
    [min_price_dd, max_price_dd, fuel_dd, body_dd, rating_dd, search_btn, output],
    layout=center_layout
)


# Display controls + output
display(ui)
